# Producción: Glomerulus Detection & Classification

End-to-end inference pipeline:
1. **U-Net (ResNet34)**: Segment glomeruli from WSI tiles (binary segmentation)
2. **EfficientNet-B2**: Classify each detected glomerulus into 4 pathological classes
3. **Metrics**: Compute segmentation and classification metrics on-the-fly
4. **Output**: GeoJSON (native WSI coordinates), crops, summary CSV

**Key constraints:**
- Tile-by-tile processing (NO canvas fusion — prevents 11GB memory error)
- Calculate segmentation metrics on-the-fly, delete arrays immediately
- Watershed for instance separation
- Reinhard normalization (BGR) before U-Net
- Z-score normalization before classification

In [1]:
# Cell 0: Imports
import torch
import torch.nn.functional as F
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm
import json
import warnings
warnings.filterwarnings('ignore')

# Segmentation & metrics
from scipy import ndimage
from scipy.ndimage import label, find_objects, distance_transform_edt
from skimage import morphology
from skimage.segmentation import watershed
from sklearn.metrics import jaccard_score, f1_score, precision_score, recall_score

# Models
import segmentation_models_pytorch as smp
import timm
from torchvision import transforms

# Image I/O
from PIL import Image
import pandas as pd

print("Imports complete.")

Imports complete.


In [2]:
# Cell 1: Configuration

# Paths
PROJECT_ROOT = Path('.')
UNET_CKPT = PROJECT_ROOT / 'checkpoints' / 'unet_binary_20260514_223037_best.pth'
CLASSIFIER_CKPT = PROJECT_ROOT / 'Salidas' / 'Clasificador' / 'best_model_default.pth'
TEST_DATA_DIR = PROJECT_ROOT / 'Salidas' / 'Imagen'
OUTPUT_DIR = PROJECT_ROOT / 'Salidas' / 'produccion_output'
GLOM_CROPS_DIR = PROJECT_ROOT / 'Salidas' / 'produccion_crops'

# Create output directories
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GLOM_CROPS_DIR.mkdir(parents=True, exist_ok=True)

# Model hyperparameters
UNET_INPUT_SIZE = 288
TILE_SIZE = 1024
ZOOM_SCALE = 0.5

# Glomerulus filtering
GLOM_MIN_AREA = 1500

# Classification
CLASS_NAMES = ['No_Proliferativo', 'Proliferativo', 'Esclerosado', 'Excluido']
NUM_CLASSES = len(CLASS_NAMES)

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")
print(f"Test data dir: {TEST_DATA_DIR}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Crops dir: {GLOM_CROPS_DIR}")

Using device: cuda
Test data dir: Salidas\Imagen
Output dir: Salidas\produccion_output
Crops dir: Salidas\produccion_crops


In [3]:
# Cell 2: Reinhard Normalization & Model Loading

class ReinhardNormalize:
    """
    Reinhard color normalization for H&E stained images.
    Takes BGR image, applies on tissue pixels only, returns BGR.
    """
    def __init__(self):
        self.target_means = np.array([0.485, 0.456, 0.406])
        self.target_stds = np.array([0.229, 0.224, 0.225])

    def __call__(self, image_bgr):
        """
        Args:
            image_bgr: numpy array, shape (H, W, 3), dtype uint8, BGR format
        Returns:
            normalized_bgr: numpy array, shape (H, W, 3), dtype uint8, BGR format
        """
        # Convert to float32 [0, 1]
        img = image_bgr.astype(np.float32) / 255.0
        
        # Convert BGR to RGB for processing
        img_rgb = cv2.cvtColor((img * 255).astype(np.uint8), cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        
        # Create tissue mask (exclude white background)
        gray = cv2.cvtColor((img_rgb * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
        tissue_mask = gray < 0.9
        
        if tissue_mask.sum() < 100:
            # Not enough tissue, return original
            return image_bgr
        
        # Compute means and stds on tissue
        means = np.array([img_rgb[tissue_mask, i].mean() for i in range(3)])
        stds = np.array([img_rgb[tissue_mask, i].std() for i in range(3)])
        stds = np.maximum(stds, 1e-6)  # Avoid division by zero
        
        # Normalize
        img_norm = img_rgb.copy()
        for i in range(3):
            img_norm[:, :, i] = (img_rgb[:, :, i] - means[i]) / stds[i] * self.target_stds[i] + self.target_means[i]
        
        # Clip and convert back to BGR
        img_norm = np.clip(img_norm, 0, 1)
        img_norm_uint8 = (img_norm * 255).astype(np.uint8)
        img_bgr_out = cv2.cvtColor(img_norm_uint8, cv2.COLOR_RGB2BGR)
        
        return img_bgr_out


def build_unet():
    """Build U-Net model with ResNet34 encoder."""
    model = smp.Unet(
        encoder_name='resnet34',
        encoder_weights='imagenet',
        in_channels=3,
        classes=1,
        activation=None
    )
    return model


def build_classifier():
    """Build EfficientNet-B2 classifier with 4-channel input and 4 output classes."""
    model = timm.create_model('efficientnet_b2', pretrained=False, in_chans=4, num_classes=NUM_CLASSES)
    return model


def load_models():
    """Load both models from checkpoints."""
    # U-Net
    unet = build_unet().to(DEVICE)
    assert UNET_CKPT.exists(), f"U-Net checkpoint not found: {UNET_CKPT}"
    ckpt = torch.load(UNET_CKPT, map_location=DEVICE)
    if 'model_state_dict' in ckpt:
        unet.load_state_dict(ckpt['model_state_dict'])
    else:
        unet.load_state_dict(ckpt)
    unet.eval()
    print(f"U-Net loaded from {UNET_CKPT}")
    
    # Classifier
    classifier = build_classifier().to(DEVICE)
    assert CLASSIFIER_CKPT.exists(), f"Classifier checkpoint not found: {CLASSIFIER_CKPT}"
    ckpt_clf = torch.load(CLASSIFIER_CKPT, map_location=DEVICE)
    if isinstance(ckpt_clf, dict) and 'model_state_dict' in ckpt_clf:
        classifier.load_state_dict(ckpt_clf['model_state_dict'])
    else:
        classifier.load_state_dict(ckpt_clf)
    classifier.eval()
    print(f"Classifier loaded from {CLASSIFIER_CKPT}")
    
    # Extract Reinhard normalizer instance
    reinhard_norm = ReinhardNormalize()
    
    return unet, classifier, reinhard_norm


# Load models
unet, classifier, reinhard_norm = load_models()
print("Models loaded successfully.")

U-Net loaded from checkpoints\unet_binary_20260514_223037_best.pth
Classifier loaded from Salidas\Clasificador\best_model_default.pth
Models loaded successfully.


In [4]:
# Cell 3: Helper Functions

import re

def parse_tile_name(tile_filename):
    """
    Parse tile filename to extract coordinates.
    Expected format: {biopsy}_tile_x{x}_y{y}_endx{endx}_endy{endy}.png
    Returns: (x, y, endx, endy) or None if parse fails
    """
    pattern = r'(.+)_tile_x(\d+)_y(\d+)_endx(\d+)_endy(\d+)'
    match = re.match(pattern, tile_filename.replace('.png', ''))
    if match:
        return int(match.group(2)), int(match.group(3)), int(match.group(4)), int(match.group(5))
    return None


def preprocess_tile_for_unet(tile_bgr):
    """
    Preprocess tile for U-Net:
    1. Reinhard normalization (BGR -> BGR)
    2. Resize to UNET_INPUT_SIZE
    3. Z-score normalization
    4. Convert to torch tensor
    """
    # Reinhard
    tile_norm = reinhard_norm(tile_bgr)
    
    # Resize
    tile_resized = cv2.resize(tile_norm, (UNET_INPUT_SIZE, UNET_INPUT_SIZE), interpolation=cv2.INTER_LINEAR)
    
    # Convert BGR to RGB for torch models
    tile_rgb = cv2.cvtColor(tile_resized, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    
    # Z-score normalization
    mean = tile_rgb.mean()
    std = tile_rgb.std() + 1e-6
    tile_normalized = (tile_rgb - mean) / std
    
    # To tensor (C, H, W)
    tile_tensor = torch.from_numpy(tile_normalized).permute(2, 0, 1).unsqueeze(0).float()
    return tile_tensor


def extract_glomeruli_watershed(pred_mask):
    """
    Extract individual glomeruli from segmentation mask using watershed.
    Args:
        pred_mask: binary mask, shape (H, W), values 0/1
    Returns:
        labeled_mask: labeled instances, shape (H, W)
        bboxes: list of (y_min, x_min, y_max, x_max) for each instance
    """
    # Morphological operations
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    pred_mask_uint8 = (pred_mask * 255).astype(np.uint8)
    pred_clean = cv2.morphologyEx(pred_mask_uint8, cv2.MORPH_CLOSE, kernel, iterations=2)
    pred_clean = cv2.morphologyEx(pred_clean, cv2.MORPH_OPEN, kernel, iterations=1)
    
    # Watershed
    dist = distance_transform_edt(pred_clean > 0)
    peaks = ndimage.maximum_filter(dist, size=15) == dist
    markers, num_markers = label(peaks)
    
    labeled_mask = watershed(-dist, markers, mask=(pred_clean > 0))
    
    # Extract bounding boxes
    bboxes = []
    for region_label in range(1, labeled_mask.max() + 1):
        region = (labeled_mask == region_label)
        area = region.sum()
        
        if area < GLOM_MIN_AREA:
            continue
        
        coords = np.argwhere(region)
        y_min, x_min = coords.min(axis=0)
        y_max, x_max = coords.max(axis=0)
        bboxes.append({
            'label': region_label,
            'y_min': int(y_min),
            'x_min': int(x_min),
            'y_max': int(y_max),
            'x_max': int(x_max),
            'area': int(area)
        })
    
    return labeled_mask, bboxes


def compute_segmentation_metrics(pred_mask, gt_mask):
    """
    Compute per-tile segmentation metrics.
    Args:
        pred_mask: predicted binary mask, shape (H, W), values 0/1
        gt_mask: ground truth mask, shape (H, W), values 0/1
    Returns:
        dict with IoU, F1, Precision, Recall
    """
    # Flatten for metric computation
    pred_flat = pred_mask.flatten()
    gt_flat = gt_mask.flatten()
    
    iou = jaccard_score(gt_flat, pred_flat, zero_division=0)
    f1 = f1_score(gt_flat, pred_flat, zero_division=0)
    prec = precision_score(gt_flat, pred_flat, zero_division=0)
    rec = recall_score(gt_flat, pred_flat, zero_division=0)
    
    return {
        'iou': float(iou),
        'f1': float(f1),
        'precision': float(prec),
        'recall': float(rec)
    }


def extract_crop_and_classify(tile_bgr, bbox, tile_coords, tile_name):
    """
    Extract glomerulus crop, classify it, save to disk.
    Args:
        tile_bgr: original tile in BGR
        bbox: dict with y_min, x_min, y_max, x_max, label, area
        tile_coords: (x_tile, y_tile, endx_tile, endy_tile) in WSI coordinates
        tile_name: original tile filename (without .png)
    Returns:
        dict with crop metadata and classification result
    """
    y_min, x_min = bbox['y_min'], bbox['x_min']
    y_max, x_max = bbox['y_max'], bbox['x_max']
    
    # Extract crop (with padding)
    pad = 10
    y_min_pad = max(0, y_min - pad)
    x_min_pad = max(0, x_min - pad)
    y_max_pad = min(tile_bgr.shape[0], y_max + pad)
    x_max_pad = min(tile_bgr.shape[1], x_max + pad)
    
    crop_bgr = tile_bgr[y_min_pad:y_max_pad, x_min_pad:x_max_pad].copy()
    
    # Apply Reinhard for crop image
    crop_bgr_normalized = reinhard_norm(crop_bgr)
    crop_rgb_normalized = cv2.cvtColor(crop_bgr_normalized, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    
    # Create mask for this crop (binary)
    crop_h, crop_w = crop_bgr.shape[:2]
    mask = np.zeros((crop_h, crop_w), dtype=np.uint8)
    mask[y_min - y_min_pad:y_max - y_min_pad, x_min - x_min_pad:x_max - x_min_pad] = 255
    
    # Save crop image and mask
    biopsy_name = tile_name.rsplit('_tile_', 1)[0]
    crop_images_dir = GLOM_CROPS_DIR / biopsy_name / 'images'
    crop_masks_dir = GLOM_CROPS_DIR / biopsy_name / 'masks'
    crop_images_dir.mkdir(parents=True, exist_ok=True)
    crop_masks_dir.mkdir(parents=True, exist_ok=True)
    
    crop_id = f"{tile_name}_glom_{bbox['label']}"
    crop_image_path = crop_images_dir / f"{crop_id}.png"
    crop_mask_path = crop_masks_dir / f"{crop_id}.png"
    
    # Save as uint8 PNG with Reinhard applied
    cv2.imwrite(str(crop_image_path), crop_bgr_normalized)
    cv2.imwrite(str(crop_mask_path), mask)
    
    # Classify: resize crop to expected input size, apply Z-score
    crop_rgb_resized = cv2.resize(crop_rgb_normalized, (128, 128), interpolation=cv2.INTER_LINEAR)
    crop_rgb_resized = np.clip(crop_rgb_resized, 0, 1)
    
    # Z-score on crop
    mean_crop = crop_rgb_resized.mean()
    std_crop = crop_rgb_resized.std() + 1e-6
    crop_rgb_zscore = (crop_rgb_resized - mean_crop) / std_crop
    
    # 4-channel input: RGB + mask
    mask_resized = cv2.resize(mask / 255.0, (128, 128), interpolation=cv2.INTER_LINEAR)
    crop_4ch = np.concatenate([crop_rgb_zscore, mask_resized[:, :, None]], axis=2)
    crop_tensor = torch.from_numpy(crop_4ch).permute(2, 0, 1).unsqueeze(0).float().to(DEVICE)
    
    # Inference
    with torch.no_grad():
        logits = classifier(crop_tensor)
        probs = F.softmax(logits, dim=1)
        pred_class = probs.argmax(dim=1).item()
        pred_prob = probs[0, pred_class].item()
    
    # WSI native coordinates for this glomerulus
    # Tile filename coords (x_tile, y_tile) are already in native WSI space
    # Bboxes are in tile pixel space (1024x1024), so direct addition
    x_tile, y_tile, endx_tile, endy_tile = tile_coords
    glom_x_min_native = x_tile + int(x_min_pad)
    glom_y_min_native = y_tile + int(y_min_pad)
    glom_x_max_native = x_tile + int(x_max_pad)
    glom_y_max_native = y_tile + int(y_max_pad)
    
    return {
        'crop_id': crop_id,
        'crop_image_path': str(crop_image_path),
        'crop_mask_path': str(crop_mask_path),
        'class': CLASS_NAMES[pred_class],
        'class_id': pred_class,
        'confidence': pred_prob,
        'tile_name': tile_name,
        'tile_coords': {'x': x_tile, 'y': y_tile, 'endx': endx_tile, 'endy': endy_tile},
        'bbox_tile': {'y_min': y_min, 'x_min': x_min, 'y_max': y_max, 'x_max': x_max},
        'bbox_native': {
            'x_min': glom_x_min_native,
            'y_min': glom_y_min_native,
            'x_max': glom_x_max_native,
            'y_max': glom_y_max_native
        },
        'area_pixels': bbox['area']
    }


print("Helper functions defined.")

Helper functions defined.


In [5]:
# Cell 4: Main Inference Loop (Tile-by-Tile)

# Collect all results
all_results = {}  # {biopsy: {glomeruli: [...], seg_metrics: [...]}}
total_glomeruli = 0

# Get list of biopsies
biopsies = sorted([d for d in TEST_DATA_DIR.iterdir() if d.is_dir()])
print(f"Found {len(biopsies)} biopsies.\n")

for biopsy_dir in biopsies:
    biopsy_name = biopsy_dir.name
    images_dir = biopsy_dir / 'images'
    masks_dir = biopsy_dir / 'masks'
    
    if not images_dir.exists() or not masks_dir.exists():
        print(f"Skipping {biopsy_name}: missing images or masks directory.")
        continue
    
    # Collect tiles
    tile_files = sorted([f for f in images_dir.glob('*.png')])
    
    if len(tile_files) == 0:
        print(f"Skipping {biopsy_name}: no tiles found.")
        continue
    
    print(f"Processing {biopsy_name}: {len(tile_files)} tiles")
    
    glomeruli_data = []
    seg_metrics = []  # List of dicts with metrics, NOT arrays
    
    for tile_file in tqdm(tile_files, desc=biopsy_name):
        tile_name = tile_file.stem
        
        # Parse coordinates
        coords = parse_tile_name(tile_name)
        if coords is None:
            print(f"  Warning: Failed to parse {tile_name}")
            continue
        
        x_tile, y_tile, endx_tile, endy_tile = coords
        
        # Load tile
        tile_bgr = cv2.imread(str(tile_file), cv2.IMREAD_COLOR)
        if tile_bgr is None:
            print(f"  Warning: Failed to load {tile_file}")
            continue
        
        # Preprocess for U-Net
        tile_tensor = preprocess_tile_for_unet(tile_bgr)
        tile_tensor = tile_tensor.to(DEVICE)
        
        # U-Net inference
        with torch.no_grad():
            logits = unet(tile_tensor)  # (1, 1, 288, 288)
            pred_sigmoid = torch.sigmoid(logits).cpu().numpy()  # (1, 1, 288, 288)
        
        # Resize mask back to tile size
        pred_mask_resized = pred_sigmoid[0, 0]  # (288, 288)
        pred_mask = cv2.resize(pred_mask_resized, (TILE_SIZE, TILE_SIZE), interpolation=cv2.INTER_LINEAR)
        pred_mask = (pred_mask > 0.5).astype(np.uint8)
        
        # GT segmentation comparison (per-tile) — compute metrics on-the-fly, discard arrays
        gt_mask_file = masks_dir / f"{tile_name}_mask.png"
        if gt_mask_file.exists():
            gt_mask_img = cv2.imread(str(gt_mask_file), cv2.IMREAD_GRAYSCALE)
            # GT values: 64=class0, 128=class1, 192=class2, 255=class3 -> binary: any non-zero = 1
            gt_mask_binary = (gt_mask_img > 0).astype(np.uint8)
            
            # Calculate metrics on-the-fly
            iou = jaccard_score(gt_mask_binary.flatten(), pred_mask.flatten(), zero_division=0)
            f1 = f1_score(gt_mask_binary.flatten(), pred_mask.flatten(), zero_division=0)
            prec = precision_score(gt_mask_binary.flatten(), pred_mask.flatten(), zero_division=0)
            rec = recall_score(gt_mask_binary.flatten(), pred_mask.flatten(), zero_division=0)
            
            seg_metrics.append({'iou': float(iou), 'f1': float(f1), 'prec': float(prec), 'rec': float(rec)})
            
            # Arrays are now garbage-collected
            del gt_mask_img, gt_mask_binary
        
        # Extract glomeruli using watershed
        labeled_mask, bboxes = extract_glomeruli_watershed(pred_mask)
        
        # For each glomerulus, extract crop and classify
        for bbox in bboxes:
            try:
                glom_data = extract_crop_and_classify(
                    tile_bgr,
                    bbox,
                    coords,
                    tile_name
                )
                glomeruli_data.append(glom_data)
                total_glomeruli += 1
            except Exception as e:
                print(f"    Error classifying glomerulus: {e}")
                continue
        
        # Clean up
        del tile_bgr, tile_tensor, pred_mask_resized, pred_mask, labeled_mask
    
    # Store results for this biopsy
    all_results[biopsy_name] = {
        'glomeruli': glomeruli_data,
        'seg_metrics': seg_metrics  # List of dicts, NOT arrays
    }
    
    print(f"  -> {len(glomeruli_data)} glomeruli detected, {len(seg_metrics)} tiles processed.\n")

print(f"\n{'='*60}")
print(f"INFERENCE COMPLETE: {total_glomeruli} glomeruli detected across all biopsies.")
print(f"{'='*60}")

Found 11 biopsies.

Processing 29-10399: 269 tiles


29-10399: 100%|██████████| 269/269 [03:10<00:00,  1.41it/s]


  -> 53 glomeruli detected, 269 tiles processed.

Processing 29-10662: 215 tiles


29-10662: 100%|██████████| 215/215 [02:25<00:00,  1.48it/s]


  -> 12 glomeruli detected, 215 tiles processed.

Processing 933-10155: 76 tiles


...

  -> 396 glomeruli detected, 329 tiles processed.

Processing BR-46-PAS-24-CONV: 118 tiles


BR-46-PAS-24-CONV: 100%|██████████| 118/118 [01:32<00:00,  1.28it/s]


  -> 255 glomeruli detected, 118 tiles processed.

Processing BR-69-PAS-24-CONV: 591 tiles


BR-69-PAS-24-CONV: 100%|██████████| 591/591 [06:55<00:00,  1.42it/s]

  -> 156 glomeruli detected, 591 tiles processed.


INFERENCE COMPLETE: 2632 glomeruli detected across all biopsies.


In [6]:
# Cell 5: Segmentation Metrics Summary

print("\n" + "="*60)
print("SEGMENTATION METRICS")
print("="*60 + "\n")

seg_metrics_summary = []

for biopsy_name, results in all_results.items():
    seg_metrics = results['seg_metrics']
    
    if len(seg_metrics) == 0:
        print(f"{biopsy_name}: No GT masks found.")
        continue
    
    # Compute per-biopsy averages from lightweight dicts
    iou_list = [m['iou'] for m in seg_metrics]
    f1_list = [m['f1'] for m in seg_metrics]
    prec_list = [m['prec'] for m in seg_metrics]
    rec_list = [m['rec'] for m in seg_metrics]
    
    iou_mean = np.mean(iou_list)
    f1_mean = np.mean(f1_list)
    prec_mean = np.mean(prec_list)
    rec_mean = np.mean(rec_list)
    
    seg_metrics_summary.append({
        'biopsy': biopsy_name,
        'num_tiles': len(seg_metrics),
        'iou': iou_mean,
        'f1': f1_mean,
        'precision': prec_mean,
        'recall': rec_mean
    })
    
    print(f"{biopsy_name}:")
    print(f"  Tiles with GT: {len(seg_metrics)}")
    print(f"  IoU (mean):       {iou_mean:.4f}")
    print(f"  F1 (mean):        {f1_mean:.4f}")
    print(f"  Precision (mean): {prec_mean:.4f}")
    print(f"  Recall (mean):    {rec_mean:.4f}\n")

# Macro-average across all biopsies
if len(seg_metrics_summary) > 0:
    df_seg = pd.DataFrame(seg_metrics_summary)
    print(f"MACRO-AVERAGE (across {len(seg_metrics_summary)} biopsies):")
    print(f"  IoU:       {df_seg['iou'].mean():.4f} (+/- {df_seg['iou'].std():.4f})")
    print(f"  F1:        {df_seg['f1'].mean():.4f} (+/- {df_seg['f1'].std():.4f})")
    print(f"  Precision: {df_seg['precision'].mean():.4f} (+/- {df_seg['precision'].std():.4f})")
    print(f"  Recall:    {df_seg['recall'].mean():.4f} (+/- {df_seg['recall'].std():.4f})\n")
    
    # Save segmentation metrics to CSV
    seg_metrics_csv = OUTPUT_DIR / 'segmentation_metrics.csv'
    df_seg.to_csv(seg_metrics_csv, index=False)
    print(f"Saved: {seg_metrics_csv}")
else:
    print("No GT masks found in test set.")


SEGMENTATION METRICS

29-10399:
  Tiles with GT: 269
  IoU (mean):       0.0474
  F1 (mean):        0.0545
  Precision (mean): 0.0711
  Recall (mean):    0.0536

29-10662:
  Tiles with GT: 215
  IoU (mean):       0.0008
  F1 (mean):        0.0014
  Precision (mean): 0.0046
  Recall (mean):    0.0008

933-10155:
  Tiles with GT: 76
  IoU (mean):       0.0578
  F1 (mean):        0.0633
  Precision (mean): 0.0757
  Recall (mean):    0.0608

BR-009-PAS-25-CONV:
  Tiles with GT: 345
  IoU (mean):       0.1109
  F1 (mean):        0.1245
  Precision (mean): 0.1488
  Recall (mean):    0.1198

BR-023-PAS-25-CONV:
  Tiles with GT: 565
  IoU (mean):       0.0592
  F1 (mean):        0.0689
  Precision (mean): 0.0897
  Recall (mean):    0.0695

BR-054-PAS-CILINDRO-1-25-CONV:
  Tiles with GT: 228
  IoU (mean):       0.1274
  F1 (mean):        0.1443
  Precision (mean): 0.1915
  Recall (mean):    0.1313

BR-064-PAS-25 SE ESCANEO DE NUEVO-CONV:
  Tiles with GT: 502
  IoU (mean):       0.0399
  F1 (me

In [7]:
# Cell 6: Classification Metrics Summary

print("\n" + "="*60)
print("CLASSIFICATION METRICS")
print("="*60 + "\n")

clf_metrics_summary = []
all_classes = []

for biopsy_name, results in all_results.items():
    glomeruli = results['glomeruli']
    
    if len(glomeruli) == 0:
        print(f"{biopsy_name}: No glomeruli detected.")
        continue
    
    # Class distribution
    class_counts = {cls: 0 for cls in CLASS_NAMES}
    for glom in glomeruli:
        class_counts[glom['class']] += 1
        all_classes.append(glom['class'])
    
    total = sum(class_counts.values())
    
    print(f"{biopsy_name}:")
    print(f"  Total glomeruli: {total}")
    for cls in CLASS_NAMES:
        count = class_counts[cls]
        pct = 100.0 * count / total if total > 0 else 0
        print(f"    {cls}: {count} ({pct:.1f}%)")
    print()
    
    clf_metrics_summary.append({
        'biopsy': biopsy_name,
        'num_glomeruli': total,
        **class_counts
    })

# Global class distribution
if len(all_classes) > 0:
    print(f"GLOBAL CLASS DISTRIBUTION (all {len(all_classes)} glomeruli):")
    class_counts_global = {cls: all_classes.count(cls) for cls in CLASS_NAMES}
    for cls in CLASS_NAMES:
        count = class_counts_global[cls]
        pct = 100.0 * count / len(all_classes)
        print(f"  {cls}: {count} ({pct:.1f}%)")
    
    # Save classification metrics to CSV
    df_clf = pd.DataFrame(clf_metrics_summary)
    clf_metrics_csv = OUTPUT_DIR / 'classification_metrics.csv'
    df_clf.to_csv(clf_metrics_csv, index=False)
    print(f"\nSaved: {clf_metrics_csv}")
else:
    print("No glomeruli detected.")


CLASSIFICATION METRICS

29-10399:
  Total glomeruli: 53
    No_Proliferativo: 35 (66.0%)
    Proliferativo: 6 (11.3%)
    Esclerosado: 0 (0.0%)
    Excluido: 12 (22.6%)

29-10662:
  Total glomeruli: 12
    No_Proliferativo: 10 (83.3%)
    Proliferativo: 2 (16.7%)
    Esclerosado: 0 (0.0%)
    Excluido: 0 (0.0%)

933-10155:
  Total glomeruli: 10
    No_Proliferativo: 4 (40.0%)
    Proliferativo: 4 (40.0%)
    Esclerosado: 0 (0.0%)
    Excluido: 2 (20.0%)

BR-009-PAS-25-CONV:
  Total glomeruli: 209
    No_Proliferativo: 65 (31.1%)
    Proliferativo: 94 (45.0%)
    Esclerosado: 2 (1.0%)
    Excluido: 48 (23.0%)

BR-023-PAS-25-CONV:
  Total glomeruli: 603
    No_Proliferativo: 105 (17.4%)
    Proliferativo: 172 (28.5%)
    Esclerosado: 124 (20.6%)
    Excluido: 202 (33.5%)

BR-054-PAS-CILINDRO-1-25-CONV:
  Total glomeruli: 167
    No_Proliferativo: 56 (33.5%)
    Proliferativo: 74 (44.3%)
    Esclerosado: 1 (0.6%)
    Excluido: 36 (21.6%)

BR-064-PAS-25 SE ESCANEO DE NUEVO-CONV:
  Total g

In [8]:
# Cell 7: Visualization & Crop Summary

print("\n" + "="*60)
print("GLOMERULUS CROPS SAVED")
print("="*60 + "\n")

for biopsy_name in sorted(all_results.keys()):
    results = all_results[biopsy_name]
    glomeruli = results['glomeruli']
    
    if len(glomeruli) == 0:
        continue
    
    print(f"{biopsy_name}: {len(glomeruli)} glomeruli")
    
    # List crop paths
    biopsy_crop_dir = GLOM_CROPS_DIR / biopsy_name / 'images'
    if biopsy_crop_dir.exists():
        crop_files = sorted(biopsy_crop_dir.glob('*.png'))
        print(f"  Crops saved: {len(crop_files)} files")
        print(f"  Location: {biopsy_crop_dir}\n")

print(f"All crops saved to: {GLOM_CROPS_DIR}")


GLOMERULUS CROPS SAVED

29-10399: 53 glomeruli
  Crops saved: 53 files
  Location: Salidas\produccion_crops\29-10399\images

29-10662: 12 glomeruli
  Crops saved: 12 files
  Location: Salidas\produccion_crops\29-10662\images

933-10155: 10 glomeruli
  Crops saved: 10 files
  Location: Salidas\produccion_crops\933-10155\images

BR-009-PAS-25-CONV: 209 glomeruli
  Crops saved: 209 files
  Location: Salidas\produccion_crops\BR-009-PAS-25-CONV\images

BR-023-PAS-25-CONV: 603 glomeruli
  Crops saved: 603 files
  Location: Salidas\produccion_crops\BR-023-PAS-25-CONV\images

BR-054-PAS-CILINDRO-1-25-CONV: 167 glomeruli
  Crops saved: 167 files
  Location: Salidas\produccion_crops\BR-054-PAS-CILINDRO-1-25-CONV\images

BR-064-PAS-25 SE ESCANEO DE NUEVO-CONV: 185 glomeruli
  Crops saved: 185 files
  Location: Salidas\produccion_crops\BR-064-PAS-25 SE ESCANEO DE NUEVO-CONV\images

BR-104-PAS-25-CONV: 586 glomeruli
  Crops saved: 586 files
  Location: Salidas\produccion_crops\BR-104-PAS-25-CONV\i

In [9]:
# Cell 8: GeoJSON Export (Native WSI Coordinates)

print("\n" + "="*60)
print("GEOJSON EXPORT")
print("="*60 + "\n")

features = []

for biopsy_name, results in all_results.items():
    glomeruli = results['glomeruli']
    
    for glom in glomeruli:
        bbox_native = glom['bbox_native']
        
        # GeoJSON feature with Rectangle geometry (using native coordinates)
        feature = {
            'type': 'Feature',
            'properties': {
                'biopsy': biopsy_name,
                'crop_id': glom['crop_id'],
                'class': glom['class'],
                'class_id': glom['class_id'],
                'confidence': glom['confidence'],
                'area_pixels': glom['area_pixels'],
                'tile_name': glom['tile_name']
            },
            'geometry': {
                'type': 'Polygon',
                'coordinates': [[
                    [bbox_native['x_min'], bbox_native['y_min']],
                    [bbox_native['x_max'], bbox_native['y_min']],
                    [bbox_native['x_max'], bbox_native['y_max']],
                    [bbox_native['x_min'], bbox_native['y_max']],
                    [bbox_native['x_min'], bbox_native['y_min']]
                ]]
            }
        }
        features.append(feature)

# Create GeoJSON FeatureCollection
geojson_data = {
    'type': 'FeatureCollection',
    'features': features
}

# Save GeoJSON
geojson_path = OUTPUT_DIR / 'glomeruli_annotations.geojson'
with open(geojson_path, 'w') as f:
    json.dump(geojson_data, f, indent=2)

print(f"GeoJSON saved: {geojson_path}")
print(f"  Total features: {len(features)}")
print(f"  Coordinate system: Native WSI coordinates")


GEOJSON EXPORT

GeoJSON saved: Salidas\produccion_output\glomeruli_annotations.geojson
  Total features: 2632
  Coordinate system: Native WSI coordinates


In [10]:
# Cell 9: Summary & Final Report

print("\n" + "="*80)
print("PRODUCTION INFERENCE COMPLETE")
print("="*80 + "\n")

# Summary statistics
total_biopsies = len(all_results)
total_glomeruli_final = sum(len(r['glomeruli']) for r in all_results.values())
total_tiles_processed = sum(len(r['seg_metrics']) for r in all_results.values())

print(f"SUMMARY:")
print(f"  Total biopsies processed:    {total_biopsies}")
print(f"  Total tiles processed:       {total_tiles_processed}")
print(f"  Total glomeruli detected:    {total_glomeruli_final}")
print(f"  Avg glomeruli per biopsy:    {total_glomeruli_final / total_biopsies:.1f}" if total_biopsies > 0 else "  N/A")

print(f"\nOUTPUT FILES:")
output_files = [
    ('Segmentation Metrics', OUTPUT_DIR / 'segmentation_metrics.csv'),
    ('Classification Metrics', OUTPUT_DIR / 'classification_metrics.csv'),
    ('GeoJSON Annotations', OUTPUT_DIR / 'glomeruli_annotations.geojson'),
    ('Crop Images', GLOM_CROPS_DIR / '*/images/'),
    ('Crop Masks', GLOM_CROPS_DIR / '*/masks/')
]

for file_type, path in output_files:
    if '*' in str(path):
        print(f"  {file_type}: {path}")
    elif path.exists():
        print(f"  {file_type}: {path}")

print(f"\nMAIN OUTPUT DIRECTORY: {OUTPUT_DIR}")
print(f"CROP STORAGE: {GLOM_CROPS_DIR}")

# Save results JSON for future reference
results_json = OUTPUT_DIR / 'all_results.json'
with open(results_json, 'w') as f:
    json.dump(all_results, f, indent=2)
print(f"\nFull results saved: {results_json}")

print(f"\n{'='*80}")


PRODUCTION INFERENCE COMPLETE

SUMMARY:
  Total biopsies processed:    11
  Total tiles processed:       4235
  Total glomeruli detected:    2632
  Avg glomeruli per biopsy:    239.3

OUTPUT FILES:
  Segmentation Metrics: Salidas\produccion_output\segmentation_metrics.csv
  Classification Metrics: Salidas\produccion_output\classification_metrics.csv
  GeoJSON Annotations: Salidas\produccion_output\glomeruli_annotations.geojson
  Crop Images: Salidas\produccion_crops\*\images
  Crop Masks: Salidas\produccion_crops\*\masks

MAIN OUTPUT DIRECTORY: Salidas\produccion_output
CROP STORAGE: Salidas\produccion_crops

Full results saved: Salidas\produccion_output\all_results.json

